# TabPFN-2.5 — fast-model EDA on Kaggle GPU

Reads an **untuned [TabPFN-2.5](https://github.com/PriorLabs/TabPFN)** score as a
second opinion on the *rough achievable score* for churn, alongside the untuned
LightGBM baseline (~0.91 ROC-AUC) and our tuned best (~0.92). Companion to the
"Fast-model EDA" section of `EDA.ipynb`; reuses the experiment-tracking harness and
the Kaggle conventions in `docs/kaggle_gpu_workflow.md`.

**Why Kaggle, not local.** TabPFN-2.5 handles up to ~50k rows / ~2k features on a
**GPU**; on CPU it is only practical at ≲1k rows. Our train set is 594k rows and the
local environment is CPU-only, so this runs on a Kaggle **T4** over a stratified
≤50k subsample. SHAP is intentionally skipped — TabPFN is not a tree, so it would
need a slow model-agnostic explainer; TreeSHAP on the LightGBM baseline already gives
the feature story.

**Notebook settings (right sidebar).** Accelerator → **GPU T4 x2** (not P100);
Internet → **On**; Add Input → **playground-series-s6e3**.

> ⚠️ This scaffold mirrors the committed TabM Kaggle run, but the TabPFN API calls
> are **not verified against the live environment**. Confirm `TabPFNClassifier`'s
> class name / kwargs against the installed `tabpfn` version, and **smoke-test one
> fold first** (set `n_splits=2`) before a full Save & Run All.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
# Cell 1
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["TABPFN_TOKEN"] = user_secrets.get_secret("TABPFN_TOKEN")


In [3]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)         # makes `from src.xxx import ...` resolve
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...


CWD: /kaggle/working/Predict-Customer-Churn


Updating files: 100% (132/132), done.


In [4]:
!pip install -q tabpfn        # Kaggle's GPU image already ships torch + CUDA

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())     # expect 2 on T4 x2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.6/745.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 

In [5]:
# cuda.is_available() can return True on an incompatible GPU — run a real op.
import torch
print("device:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
try:
    _ = (torch.randn(16, device="cuda") @ torch.randn(16, 16, device="cuda")).sum().item()
    print("GPU compute OK")
except Exception as e:
    print("GPU compute FAILED:", e)            # if this fails, switch to T4 and restart

from tabpfn import TabPFNClassifier
print("TabPFNClassifier imported OK")

device: Tesla T4 | count: 2
GPU compute OK
TabPFNClassifier imported OK


In [6]:
# data/processed/*.parquet are git-ignored, so absent from the clone. Rebuild them
# from the attached competition CSVs.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

import sys, os

sys.path.insert(0, REPO_ROOT)            # force the cloned repo's `src` to the front
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]                  # drop any stale/shadowing `src` from an earlier cell

import shutil
from pathlib import Path
raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(force=True)


Preprocessed and saved: train_df (594194, 42), test_df (254655, 41)


In [7]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score

from src.tracking import RUNS_DIR
from src.cv import run_cv_experiment, save_experiment

### Design matrix and TabPFN subsample

`prepare_data` returns the one-hot-encoded frames used across the project, so TabPFN
sees the same numeric features as the GBDT baselines. TabPFN-2.5 caps at ~50k context
rows, so we draw a **stratified** `SUBSAMPLE_N`-row sample (preserving the ~23% churn
rate) and hand *that* to the CV harness. Two consequences:

- The OOF ROC-AUC is computed over the subsample — a valid generalization estimate,
  just on ≤50k rows (the most TabPFN can use as context).
- Each 5-fold split fits TabPFN on ~`0.8 · SUBSAMPLE_N` rows and predicts the full
  254k-row test set; the per-fold **test** predictions over 254k rows are the slow
  part. Reduce `SUBSAMPLE_N` or `n_splits` if you hit GPU-memory or time limits.

In [8]:
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]

# TabPFN-2.5's GPU row budget. 5-fold => ~0.8 * SUBSAMPLE_N rows fit per fold.
SUBSAMPLE_N = 50_000
X_sub, _, y_sub, _ = train_test_split(
    X_train, y_train, train_size=SUBSAMPLE_N, stratify=y_train, random_state=42,
)
X_sub = X_sub.reset_index(drop=True)
y_sub = y_sub.reset_index(drop=True)
print(f"Subsample: {X_sub.shape}  churn rate: {y_sub.mean():.3f}  (full: {y_train.mean():.3f})")

Subsample: (50000, 40)  churn rate: 0.225  (full: 0.225)


### Run configuration — TabPFN-2.5 baseline

Same `run_config` shape as the TabM run, fed the subsample. `metric=accuracy_score`
mirrors the other runs; the harness *always* logs **OOF ROC-AUC** separately (the
project's primary metric), so that is the achievable-score signal we read.
`save_models=False` because TabPFN is torch-backed (fragile to `joblib.dump`).

In [9]:
tabpfn_params = {
    'device':                    'cuda',
    'ignore_pretraining_limits': True,   # allow context > the default ~10k-row cap
    'random_state':              42,
}

DATA_VERSION = 'fe_v0'   # identity FE (no engineered features) — baseline

run_config = {
    'model_factory': lambda params: TabPFNClassifier(**params),
    'params':        tabpfn_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'tabpfn-eda',
    'notes': (
        'TabPFN-2.5 (untuned) fast-EDA pass on Kaggle T4 GPU, stratified 50k subsample '
        '(TabPFN row cap). OOF ROC-AUC read as a second opinion vs untuned LGBM ~0.91. '
        'tabpfn=8.0.6, torch=2.10.0+cu128. Data regenerated on-platform — data_hash differs '
        'from local runs; GPU run not bit-reproducible. SHAP intentionally skipped.'
    ),
    'parent_run_id': '',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

In [10]:
# Step 1 - Run the experiment. SMOKE-TEST FIRST: set n_splits=2 in the cell above,
# confirm the GPU path works and check per-fold time, before a full Save & Run All.
result = run_cv_experiment(run_config, X_sub, y_sub, X_test, encoded_features)

Run ID: 20260603-211648-c10040
Tag:    tabpfn-eda



tabpfn-v3-classifier-v3_default.ckpt:   0%|          | 0.00/213M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

Fold 0: accuracy=0.8589  roc_auc=0.9148  (fit 7.0s)
Fold 1: accuracy=0.8608  roc_auc=0.9154  (fit 1.3s)
Fold 2: accuracy=0.8577  roc_auc=0.9106  (fit 1.4s)
Fold 3: accuracy=0.8575  roc_auc=0.9110  (fit 1.3s)
Fold 4: accuracy=0.8597  roc_auc=0.9121  (fit 1.3s)

OOF accuracy: 0.8589
OOF ROC-AUC:  0.9127
Folds:        0.8589 ± 0.0012

Run complete. Call save_experiment(result) to log this run permanently.


In [11]:
# Step 2 - Save the run (optional). Review the OOF ROC-AUC printed above first.
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260603-211648-c10040


### Build a submission (optional)

`test_proba_mean` is the fold-bagged churn probability for the full test set. The
competition metric is ROC-AUC, so submit the probability directly.

In [12]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

       id     Churn
0  594194  0.048797
1  594195  0.000415
2  594196  0.061492
3  594197  0.002408
4  594198  0.465232
wrote /kaggle/working/submission.csv (254655, 2)


### Artifact hand-off

To fold this run back into the local repo — extract the run dir, append the new
`runs.csv` row, optionally submit and backfill `lb_public` / `lb_private`, and commit
this notebook under `kaggle/` — follow **§7–8 of `docs/kaggle_gpu_workflow.md`**.
Start by zipping the run directory for download (Output tab):

    import shutil
    from src.tracking import RUNS_DIR
    shutil.make_archive(f"/kaggle/working/{run_id}", "zip", RUNS_DIR / run_id)

In [13]:
import shutil
from src.tracking import RUNS_DIR
shutil.make_archive(f"/kaggle/working/{run_id}", "zip", RUNS_DIR / run_id)

'/kaggle/working/20260603-211648-c10040.zip'